# Image Retrieval

- This notebook contains code to carry out image retrieval based on the CLIP Image Embeddings generated for the LAION-5B dataset.

## Non-Faiss Exact Cosine Similarity Image Search

This function searches a dataset of precomputed CLIP embeddings to find images most similar to a given text prompt, an image, or a combination of both. It first computes normalized embeddings for the query using the specified CLIP model, then iterates through the dataset stored in Parquet files, loading embeddings in manageable chunks. For each chunk, it calculates cosine similarities between the query and dataset embeddings, combines text and image similarities using a weighted alpha parameter, filters results based on an optional similarity threshold, and collects metadata such as URLs, captions, and original image indices. Finally, it sorts all matches by similarity and returns the top results if requested.

In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch
from typing import List, Optional
import numpy as np
import pyarrow.dataset as ds
import os
import re
import pyarrow.parquet as pq

def find_similar_images(
    dataset_dir: str,
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,
    similarity_threshold: Optional[float] = None,
    alpha: float = 0.5
) -> List[dict]:
    ''' 
    Functions used to find images similar to a text or image (or both) based on the CLIP embeddings.

    Inputs:
    - dataset_dir: Directory containing the CLIP embeddings dataset in Parquet format.
    - model_name: Name of the pre-trained CLIP model to use.
    - text_prompt: Optional text prompt to find similar images. (None means only the image is used to search).
    - image_path: Optional path to an image to find similar images. (None means only the text prompt is used to search).
    - top_n: Optional number of top similar images to return. (None means return all).
    - similarity_threshold: Optional threshold for cosine similarity to filter results. (None means no filtering).
    - alpha: Weight for text similarity in the combined similarity score. (0.5 - equal weight between text and image, 1 - text prompt only is used, 0 - image only is used).

    Outputs:
    - List of dictionaries containing URLs, captions, original image indices, and cosine similarities of the most similar images.
    '''

    assert text_prompt or image_path, "You must provide either a text prompt or an image path."

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text_embedding = None
    image_embedding = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy for similarity calculation
    text_embedding_np = text_embedding.cpu().squeeze().numpy() if text_embedding is not None else None
    image_embedding_np = image_embedding.cpu().squeeze().numpy() if image_embedding is not None else None

    all_matches = []

    # Loads only matching files from dataset_dir like part-00000.parquet to avoid loading checkpoints
    pattern = re.compile(r"^part-\d{5}\.parquet$")
    valid_files = [
        os.path.join(dataset_dir, f)
        for f in os.listdir(dataset_dir)
        if pattern.match(f)
    ]
    dataset = ds.dataset(valid_files, format="parquet")

    # Iterate through each fragment (.parquet file) composing the dataset
    for fragment in dataset.get_fragments():
        print(f"Processing fragment: {fragment.path}")
        pq_file = pq.ParquetFile(os.path.join(dataset_dir, fragment.path))

        for row_group_index in range(pq_file.num_row_groups):
            table = pq_file.read_row_group(row_group_index)
            df_chunk = table.to_pandas()

            df_chunk = df_chunk.dropna(subset=['embeddings_result'])
            if df_chunk.empty:
                continue

            df_chunk['embeddings_result'] = df_chunk['embeddings_result'].apply(
                lambda x: np.array(x) if isinstance(x, list) else x
            )
            image_embeddings = np.vstack(df_chunk['embeddings_result'].values)

            # Compute similarities separately
            sim_text = np.dot(image_embeddings, text_embedding_np.T) if text_embedding_np is not None else 0
            sim_image = np.dot(image_embeddings, image_embedding_np.T) if image_embedding_np is not None else 0

            # Combine similarity with weights
            combined_sim = None
            if text_embedding_np is not None and image_embedding_np is not None:
                combined_sim = alpha * sim_text + (1 - alpha) * sim_image
            elif text_embedding_np is not None:
                combined_sim = sim_text
            else:
                combined_sim = sim_image

            df_chunk['combined_similarity'] = combined_sim

            if similarity_threshold is not None:
                df_chunk = df_chunk[df_chunk['combined_similarity'] >= similarity_threshold]

            for _, row in df_chunk.iterrows():
                all_matches.append({
                    'url': row.get('url'),
                    'caption': row.get('caption'),
                    'original_image_index': row.get('original_image_index'),
                    'cosine_similarity': row['combined_similarity']
                })

    all_matches.sort(key=lambda x: x['cosine_similarity'], reverse=True)
    top_matches = all_matches[:top_n] if top_n is not None else all_matches

    return top_matches

In [ ]:
dataset_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "umpire"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 
top_n = 10
similarity_threshold = 0.15

image_info_list = find_similar_images(
    dataset_dir=dataset_dir,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path = search_image_path,
    top_n=top_n,
    similarity_threshold=similarity_threshold,
    alpha=1  # 1 = text only, 0 = image only, 0.5 = equal weighting
)

# image_info_list

Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00000.parquet


In [ ]:
len(image_info_list)

1000

## Faiss Exact Cosine Similarity Image Search

* **`build_faiss_index_with_mapping_resume_exact` function:**
  This function builds or resumes a FAISS index from a list of Parquet files containing precomputed embeddings. It reads embeddings in batches, normalizes them for cosine similarity, and adds them to an exact `IndexFlatIP` FAISS index. The function supports resuming from an existing index and mapping, skipping already indexed embeddings. It maintains a mapping between vector indices and image paths, and periodically saves both the FAISS index and mapping to disk to prevent data loss. It processes large datasets efficiently by iterating over batches and shards, ensuring that even very large embeddings can be indexed without exceeding memory limits.

* **`search_sharded_faiss` function:**
  This function performs an exact similarity search across multiple FAISS shard indexes, derived from the prior function. It computes normalized embeddings for a given text prompt and/or image using a CLIP model, optionally combining them with a weighted alpha parameter. The function then streams through each FAISS shard, performing inner-product searches to retrieve the most similar images, applying an optional similarity threshold. Results from all shards are aggregated, sorted by similarity, and truncated to the top-k matches if requested. This approach allows searching very large embedding collections without loading all data into memory at once.


In [1]:
import faiss
import numpy as np
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import ast
import pickle
from tqdm import tqdm
import os
import pyarrow.parquet as pq
import pickle, ast, gc, os

def build_faiss_index_with_mapping_resume_exact(
    parquet_files: list,
    index_path: str,
    mapping_path: str,
    batch_size: int = 50_000,
    save_every: int = 100_000
):
    """
    Build or resume a FAISS index from Parquet embeddings with exact cosine similarity search.
    Uses IndexFlatIP for exact search (inner product), L2-normalized for cosine similarity.
    Skips already indexed embeddings if index/mapping exists.
    Saves index and mapping incrementally to disk.
    """
    # --- Load existing index & mapping if available ---
    if os.path.exists(index_path) and os.path.exists(mapping_path):
        print(f"Loading existing FAISS index from {index_path}")
        index = faiss.read_index(str(index_path))
        idx_to_path = pickle.load(open(str(mapping_path), "rb"))
        total_added = len(idx_to_path)
        print(f"Loaded {total_added} previously indexed vectors.")
    else:
        index = None
        idx_to_path = []
        total_added = 0

    vectors_since_last_save = 0

    def parse_embeddings_column(batch, shard_name, start_idx):
        batch = batch.filter(pc.field("embeddings_result").is_valid())
        if batch.num_rows == 0:
            return None, []

        emb_col = batch.column("embeddings_result").to_pylist()
        index_col = batch.column("original_image_index").to_pylist()

        embeddings, paths = [], []

        for i, (e, idx) in enumerate(zip(emb_col, index_col)):
            global_idx = start_idx + i
            if global_idx < len(idx_to_path):
                continue  # Already indexed
            if e is None:
                continue
            if isinstance(e, str):
                try:
                    e = ast.literal_eval(e)
                except Exception:
                    continue
            embeddings.append(np.array(e, dtype=np.float32))
            paths.append(f"{shard_name}_images/{idx}.jpg")

        if not embeddings:
            return None, []

        return np.vstack(embeddings), paths

    # --- Compute total valid embeddings already indexed ---
    total_valid_indexed = len(idx_to_path)

    # --- Count total valid embeddings in parquet files for progress bar ---
    total_valid_rows = 0
    for f in parquet_files:
        pf = pq.ParquetFile(f)
        for i in range(pf.num_row_groups):
            column_data = pf.read_row_group(i, columns=["embeddings_result"])["embeddings_result"]
            total_valid_rows += pc.count(column_data).as_py()

    remaining_valid_rows = total_valid_rows - total_valid_indexed
    if remaining_valid_rows <= 0:
        print("All embeddings are already indexed. No new vectors to add.")
        return

    total_batches_remaining = (remaining_valid_rows + batch_size - 1) // batch_size

    # --- Process batches ---
    with tqdm(total=total_batches_remaining, unit="batches", desc="Processing remaining batches") as pbar:
        for shard_path in parquet_files:
            shard_name = Path(shard_path).stem
            dataset = ds.dataset(shard_path, format="parquet")
            processed_rows = 0  # global row index within this shard

            for batch in dataset.to_batches(batch_size=batch_size):
                emb, paths = parse_embeddings_column(batch, shard_name, start_idx=processed_rows)
                processed_rows += len(batch)

                if emb is None:
                    continue  # no valid rows in this batch

                # Normalize embeddings
                faiss.normalize_L2(emb)

                # Create FAISS index if it doesn't exist
                if index is None:
                    dim = emb.shape[1]
                    index = faiss.IndexFlatIP(dim)
                    print(f"Created exact FAISS IndexFlatIP with dim={dim}")

                index.add(emb)
                idx_to_path.extend(paths)
                vectors_since_last_save += len(paths)
                total_added += len(paths)

                # Save checkpoint periodically
                if vectors_since_last_save >= save_every:
                    faiss.write_index(index, str(index_path))
                    with open(str(mapping_path), "wb") as f:
                        pickle.dump(idx_to_path, f)
                    vectors_since_last_save = 0

                pbar.update(1)

        gc.collect()

    # --- Final save ---
    faiss.write_index(index, str(index_path))
    with open(str(mapping_path), "wb") as f:
        pickle.dump(idx_to_path, f)

    print(f"FAISS index saved at {index_path}")
    print(f"Mapping saved at {mapping_path}")
    print(f"Total vectors in index: {len(idx_to_path)}")


In [ ]:
import os
from pathlib import Path

# List of embedding directories
dataset_dirs = [
    r"F:\Thesis\0000_embeddings_cleaned",
    r"F:\Thesis\0001_embeddings_cleaned",
    r"F:\Thesis\0002_embeddings_cleaned",
    r"F:\Thesis\0003_embeddings_cleaned",
]

output_base_dir = r"E:\Thesis\image_retrieval_faiss_indices"
os.makedirs(output_base_dir, exist_ok=True)

for dataset_dir in dataset_dirs:
    all_files = os.listdir(dataset_dir)
    parquet_files = [
        os.path.join(dataset_dir, f)
        for f in all_files
        if os.path.splitext(f)[1].lower() == ".parquet"
    ]

    dataset_base = Path(dataset_dir).stem

    for parquet_path in parquet_files:
        parquet_name = Path(parquet_path).stem

        # Dynamic output paths
        output_index_path = os.path.join(output_base_dir, f"faiss_{dataset_base}_{parquet_name}_IndexFlatIP.index")
        output_mapping_path = os.path.join(output_base_dir, f"faiss_{dataset_base}_{parquet_name}_mapping.pkl")

        print(f"Processing {dataset_base}/{parquet_name}...")

        build_faiss_index_with_mapping_resume_exact(
            parquet_files=[parquet_path],
            index_path=output_index_path,
            mapping_path=output_mapping_path,
            batch_size=50_000,
            save_every=100_000
        )

        print(f"Saved index to {output_index_path} and mapping to {output_mapping_path}\n")


Processing 0003_embeddings_cleaned/part-00000...
Loading existing FAISS index from E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00000_IndexFlatIP.index
Loaded 1131230 previously indexed vectors.


Processing remaining batches: 18batches [07:20, 24.50s/batches]                  


FAISS index saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00000_IndexFlatIP.index
Mapping saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00000_mapping.pkl
Total vectors in index: 1716201
Saved index to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00000_IndexFlatIP.index and mapping to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00000_mapping.pkl

Processing 0003_embeddings_cleaned/part-00001...


Processing remaining batches:   4%|▎         | 1/28 [00:11<05:18, 11.81s/batches]

Created exact FAISS IndexFlatIP with dim=768


...

FAISS index saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00007_IndexFlatIP.index
Mapping saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00007_mapping.pkl
Total vectors in index: 1367019
Saved index to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00007_IndexFlatIP.index and mapping to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00007_mapping.pkl

Processing 0003_embeddings_cleaned/part-00008...


Processing remaining batches:  17%|█▋        | 1/6 [00:10<00:51, 10.39s/batches]

Created exact FAISS IndexFlatIP with dim=768


Processing remaining batches: 8batches [01:18,  9.82s/batches]                  


FAISS index saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index
Mapping saved at E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl
Total vectors in index: 265287
Saved index to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index and mapping to E:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl



In [ ]:
# import faiss
# import numpy as np
# from transformers import AutoProcessor, AutoModel
# from PIL import Image
# import torch
# import pickle
# from pathlib import Path
# from typing import List, Optional

# def search_sharded_faiss(
#     shard_index_paths: List[str],
#     shard_mapping_paths: List[str],
#     model_name: str,
#     text_prompt: Optional[str] = None,
#     image_path: Optional[str] = None,
#     top_k: Optional[int] = None,
#     alpha: float = 0.5,
#     similarity_threshold: Optional[float] = None,
#     device: str = None
# ) -> List[dict]:
#     """
#     Stream-search multiple FAISS shard indexes for exact cosine similarity matches.
    
#     Args:
#         shard_index_paths: List of paths to FAISS index files (IndexFlatIP).
#         shard_mapping_paths: List of corresponding mapping pickle files.
#         model_name: CLIP model for embedding computation.
#         text_prompt: Optional text query.
#         image_path: Optional image query.
#         top_k: Return top_k results.
#         alpha: Weight for text vs image embedding (0=image,1=text,0.5=equal).
#         similarity_threshold: Minimum cosine similarity to include a result.
#         device: 'cuda' or 'cpu'. Defaults to CUDA if available.
        
#     Returns:
#         List of dicts: {"image_path": ..., "score": ...}, sorted by score desc.
#     """
#     assert text_prompt or image_path, "Provide either text_prompt or image_path."

#     device = device or ("cuda" if torch.cuda.is_available() else "cpu")
#     processor = AutoProcessor.from_pretrained(model_name)
#     model = AutoModel.from_pretrained(model_name).to(device)
#     model.eval()

#     # --- Compute query embedding ---
#     text_emb, image_emb = None, None

#     if text_prompt:
#         inputs = processor(text=text_prompt, return_tensors="pt").to(device)
#         with torch.no_grad():
#             text_features = model.get_text_features(**inputs)
#         text_emb = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

#     if image_path:
#         image = Image.open(image_path).convert("RGB")
#         inputs = processor(images=image, return_tensors="pt").to(device)
#         with torch.no_grad():
#             image_features = model.get_image_features(**inputs)
#         image_emb = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

#     if text_emb is not None and image_emb is not None:
#         query_emb = alpha * text_emb + (1 - alpha) * image_emb
#     elif text_emb is not None:
#         query_emb = text_emb
#     else:
#         query_emb = image_emb

#     query_emb = query_emb.cpu().numpy().astype("float32")
#     faiss.normalize_L2(query_emb)

#     all_results = []

#     # --- Stream through each shard ---
#     for idx_path, mapping_path in zip(shard_index_paths, shard_mapping_paths):
#         print(f"Searching shard index: {idx_path}")

#         group_id = Path(idx_path).stem.split("_")[1]

#         # Load index & mapping for current shard
#         index = faiss.read_index(str(idx_path))
#         with open(mapping_path, "rb") as f:
#             idx_to_path = pickle.load(f)

#         D, I = index.search(query_emb, len(idx_to_path))

#         for score, idx in zip(D[0], I[0]):
#             if idx < 0 or idx >= len(idx_to_path):
#                 continue
#             if similarity_threshold is not None and score < similarity_threshold:
#                 continue
#             all_results.append({"image_path": idx_to_path[idx], "score": float(score), "group_id": group_id})

#         # Free memory of shard index
#         del index

#     # --- Merge and sort results across shards ---
#     all_results = sorted(all_results, key=lambda x: x["score"], reverse=True)
#     if top_k is not None:
#         all_results = all_results[:top_k]

#     return all_results


In [1]:
import faiss
import numpy as np
from transformers import AutoProcessor, AutoModel
import torch
import pickle
from pathlib import Path
from typing import List, Optional
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def search_images(
    shard_index_paths: List[str],
    shard_mapping_paths: List[str],
    model_name: str,
    prompts: List[str],
    negative_prompts: Optional[List[str]] = None,
    top_k: int = 10,
    similarity_threshold: Optional[float] = None,
    copy_images: bool = False,
    image_dir: Optional[str] = None,
    output_dir: Optional[str] = None,
    device: str = None,
    max_workers: int = 8
):
    """
    Search multiple FAISS shards for multiple profession prompts simultaneously.
    Supports optional negative prompts to penalize unwanted concepts.
    For each shard, retrieves 10x top_k results, filters by similarity_threshold if specified,
    and then merges results from all shards to return global top_k results per prompt.

    Optionally copies retrieved images to directories named by prompt using multithreading.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # Encode positive prompts
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Encode negative prompts if provided
    if negative_prompts:
        neg_inputs = processor(text=negative_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            neg_features = model.get_text_features(**neg_inputs)
        neg_features = neg_features / neg_features.norm(p=2, dim=-1, keepdim=True)
        neg_mean = neg_features.mean(dim=0, keepdim=True)
        text_features = text_features - neg_mean
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    query_embs = text_features.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_embs)

    n_queries = query_embs.shape[0]
    all_results_per_query = [[] for _ in range(n_queries)]

    # Search through all shards
    for idx_path, mapping_path in zip(shard_index_paths, shard_mapping_paths):
        group_id = Path(idx_path).stem.split("_")[1]
        print(f'Searching Group: {group_id}')

        index = faiss.read_index(str(idx_path))
        with open(mapping_path, "rb") as f:
            idx_to_path = pickle.load(f)

        per_shard_k = min(len(idx_to_path), top_k * 10)
        D, I = index.search(query_embs, per_shard_k)

        for qi in range(n_queries):
            for score, idx in zip(D[qi], I[qi]):
                if idx < 0 or idx >= len(idx_to_path):
                    continue
                if similarity_threshold is not None and score < similarity_threshold:
                    continue
                all_results_per_query[qi].append({
                    "image_path": idx_to_path[idx],
                    "score": float(score),
                    "shard": Path(idx_path).stem,
                    "group_id": group_id
                })

        del index

    # Sort and truncate globally per prompt
    for qi in range(n_queries):
        all_results_per_query[qi] = sorted(
            all_results_per_query[qi],
            key=lambda x: x["score"],
            reverse=True
        )[:top_k]

    # # Multithreaded image copying
    # if copy_images:
    #     print(f'Copying Images')
    #     if output_dir is None:
    #         raise ValueError("output_dir must be specified if copy_images=True")
    #     output_root = Path(output_dir)
    #     output_root.mkdir(parents=True, exist_ok=True)

    #     def copy_single_image(r, prompt_dir):
    #         src_path = Path(r["image_path"])
    #         if not src_path.is_absolute():
    #             src_path = Path(image_dir) / Path(src_path.parent.parent) / f"{r['group_id']}_images" / src_path.name
    #         if src_path.exists():
    #             dest_filename = f"{r['score']:.3f}_" +f"{r['group_id']}_images_"+ src_path.name
    #             shutil.copy2(src_path, prompt_dir / dest_filename)
    #         else:
    #             print(f"⚠️ Missing file: {src_path}")

    #     # Prepare thread pool
    #     tasks = []
    #     with ThreadPoolExecutor(max_workers=max_workers) as executor:
    #         for qi, prompt in enumerate(prompts):
    #             prompt_dir = output_root / prompt.replace(" ", "_")
    #             prompt_dir.mkdir(parents=True, exist_ok=True)
    #             for r in all_results_per_query[qi]:
    #                 tasks.append(executor.submit(copy_single_image, r, prompt_dir))

    #         # Optional progress bar
    #         for _ in tqdm(as_completed(tasks), total=len(tasks), desc="Copying images"):
    #             pass

    # Sequential image copying
    if copy_images:
        print(f'Copying Images')
        if output_dir is None:
            raise ValueError("output_dir must be specified if copy_images=True")
        output_root = Path(output_dir)
        output_root.mkdir(parents=True, exist_ok=True)

        def copy_single_image(r, prompt_dir):
            src_path = Path(r["image_path"])
            if not src_path.is_absolute():
                src_path = Path(image_dir) / Path(src_path.parent.parent) / f"{r['group_id']}_images" / src_path.name
            if src_path.exists():
                dest_filename = f"{r['score']:.3f}_" + f"{r['group_id']}_images_" + src_path.name
                shutil.copy2(src_path, prompt_dir / dest_filename)
            else:
                print(f"⚠️ Missing file: {src_path}")

        # Sequential loop with progress bar
        total_files = sum(len(r) for r in all_results_per_query)
        with tqdm(total=total_files, desc="Copying images", unit="img") as pbar:
            for qi, prompt in enumerate(prompts):
                prompt_dir = output_root / prompt.replace(" ", "_")
                prompt_dir.mkdir(parents=True, exist_ok=True)
                for r in all_results_per_query[qi]:
                    copy_single_image(r, prompt_dir)
                    pbar.update(1)


    # Return results
    return [
        {"prompt": prompts[i], "results": all_results_per_query[i]}
        for i in range(n_queries)
    ]

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path

# Directory containing all FAISS index and mapping files
faiss_dir = r"E:\Thesis\image_retrieval_faiss_indices"

# Collect all index and mapping files dynamically
shard_indexes = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])
shard_mappings = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

profession_list = [
    "Barber", "Coach", "Businessperson", "Football Player", "Construction Worker",
    "Manager", "CEO", "Accountant", "Commander", "Firefighter", "Mover",
    "Software Developer", "Guard", "Baker", "Doctor", "Athlete", "Artist",
    "Dancer", "Mathematician", "Janitor", "Carpenter", "Mechanic", "Actor",
    "Handyman", "Musician", "Detective", "Politician", "Entrepreneur", "Model",
    "Opera Singer", "Lawyer", "Farmer", "Writer", "Librarian", "Soldier",
    "Real-Estate Developer", "Broker", "Scientist", "Butcher", "Electrician",
    "Prosecutor", "Banker", "Cook", "Hairdresser", "Prisoner", "Plumber",
    "Boxer", "Chess Player", "Priest", "Swimmer", "Tennis Player",
    "Supervisor", "Attendant", "Housekeeper", "Maid", "Producer", "Researcher",
    "Midwife", "Judge", "Bartender", "Economist",
    "Psychologist", "Theologian", "Salesperson", "Physician", "Sheriff",
    "Cashier", "Assistant", "Receptionist", "Editor", "Engineer", "Comedian",
    "Diplomat", "Guitarist", "Linguist", "Poet",
    "Laborer", "Teacher", "Delivery Driver", "Realtor", "Pilot", "Professor",
    "Historian", "Singer", "Secretary", "Auditor", "Counselor", 
    "Designer", "Journalist", "Dentist", "Analyst", "Nurse", "Tailor", "Waiter",
    "Architect", "Illustrator", "Clerk", "Police Officer", "Chef",
    "Photographer", "Cleaner", "Pharmacist", "Pianist", "Composer", "Handball Player",
    "Sociologist", "Audiologist", "Computer Programmer", "Dietitian", "DJ", "Driver",
    "Florist", "Graphic Designer", "Magician", "Makeup Artist", "Marine Biologist",
    "Nanny", "Optician", "Pastry Chef", "Sailor", "Social Worker",
    "Statistician", "Surgeon", "Technician", "Therapist", "Tour Guide", "Translator",
    "Vet", "Videographer", "Astronaut", "Biologist", "Civil Engineer",
    "Administrative Assistant", "Building Inspector", "Crane Operator", "Announcer",
    "Drafter", "Custodian", "Roofer", "PR Person", "Veterinarian", "Lab Tech",
    "Telemarketer", "Baseball Player", "Basketball Player", "Batter",
    "Goalie", "Pitcher", "Skater", "Skier",
    "Snowboarder", "Surfer", "Paramedic", "Mountain Rescue Worker",
    "Secret Service Agent", "Bodyguard", "Lifeguard", "SWAT Officer", "Security Officer",
    "Bounty Hunter", "Air Traffic Controller", "Flight Attendant", "Rocket Scientist",
    "Airplane Pilot", "Helicopter Pilot", "Train Conductor", "Taxi Driver", "Limousine Driver",
    "Race Car Driver", "Motorcycle Courier", "Bicycle Messenger", "Ship Captain",
    "Fisherman", "Animal Trainer", "Dog Walker", "Horse Trainer", "Zookeeper",
    "Park Ranger", "Wildlife Photographer", "Conservationist", "Ornithologist", 
    "Beekeeper", "Winemaker", "Brewer", "Sommelier", "Barista", "Chocolate Maker",
    "Candy Maker", "Cake Decorator", "Ice Cream Maker", "Cheese Maker", "Butler",
    "Valet", "Chauffeur", "Hotel Concierge", "Event Planner", "Wedding Planner",
    "Museum Curator", "Art Historian", "Gallery Owner", "Auctioneer",
    "Interior Designer", "Set Designer", "Costume Designer", "Fashion Designer",
    "Tattoo Artist", "Piercer", "Sculptor", "Glassblower",
    "Ceramic Artist", "Calligrapher", "Animator", "Video Editor", "Sound Engineer",
    "Lighting Technician", "Stage Manager", "Director", "Film Producer", "Critic",
    "News Anchor", "Sports Commentator", "Referee", "Martial Arts Instructor", "Personal Trainer", "Dance Instructor",
    "Ballet Teacher", "Clown Performer", "Acrobat", "Juggler",
    "Stunt Performer", "Fire Performer", "Musical Director", "Orchestra Conductor",
    "Rap Artist", "Pop Singer", "Rock Singer", "Jazz Musician", "Pianist", "Violinist",
    "Cellist", "Flutist", "Trumpet Player", "Saxophonist", "Drummer", "Guitarist",
    "Bass Guitarist", "Keyboardist", "Content Creator", "Photographer", "Drone Operator",
    "Documentary Filmmaker", "Author",
    "Editor", "Translator", "Interpreter", "Professor", "Teacher",
    "Librarian", "Archivist", "Historian", "Anthropologist", "Archaeologist",
    "Network Engineer", "Data Scientist", "Machine Learning Engineer",
    "AI Researcher", "Physicist", "Chemist", "Biologist", "Ecologist", "Geologist",
    "Oceanographer", "Meteorologist", "Pharmacist", "Nurse",
    "Dentist", "Optometrist", "Lawyer", "Judge", "Prosecutor",
    "Soldier", "Pilot", "Chef", "Florist", "Gardener", "Logger", "Miner",
    "Welder", "Blacksmith", "Bricklayer", "Plumber", "Mechanic", "Carpenter",
    "Tailor", "Seamstress", "Professional Gamer", "Chess Player",
    "Mountain Climber", "Diver", "Sailor", "Animal Shelter Worker", "Veterinarian"
]

profession_list = profession_list[:1]

print(len(profession_list))

def generate_custom_list(object_list, template="{object}"):
    """
    Generate a list of strings by applying a template to each profession.

    Args:
        object_list (list of str): List of profession names.
        template (str): Template string using "{object}" as a placeholder for the object.

    Returns:
        list of str: List with template applied to each profession.
    """
    result = []
    for obj in object_list:
        result.append(template.format(object=obj))
    return result

prompt_templates = ["Male Person {object}", "Female Person {object}"] #, "Male Human {object}", "Female Human {object}"]

# Generate Male/Female versions
prompts = []
for template in prompt_templates:
    prompts+=generate_custom_list(profession_list, template=template)

# Remove duplicates (just in case)
prompts = list(set(prompts))
print(prompts)

results = search_images(
    shard_index_paths=shard_indexes,
    shard_mapping_paths=shard_mappings,
    model_name="openai/clip-vit-large-patch14",
    prompts=prompts,
    negative_prompts=["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
    top_k=6160, #1540*4,
    similarity_threshold=0.15,
    copy_images=True,
    image_dir=r'F:\Thesis',
    output_dir=r'E:\Thesis\image_retrieval_images'
)

1
['Female Person Barber', 'Male Human Barber', 'Male Person Barber', 'Female Human Barber']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Copying images: 100%|██████████| 15153/15153 [00:18<00:00, 817.00it/s] 


In [2]:
from tqdm import tqdm
from pathlib import Path

# Collect all index and mapping files dynamically
faiss_dir = r"E:\Thesis\image_retrieval_faiss_indices"
shard_indexes = [sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])[0]]
shard_mappings = [sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])[0]]
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

# Function to split a list into chunks
def chunk_list(lst, n):
    """Yield successive n-sized chunks from list."""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

# Parameters
chunk_size = 5
prompt_templates = ["Male Person {object}", "Female Person {object}"] #, "Male Human {object}", "Female Human {object}"]

# Generate all prompts first
profession_list = [
    "Barber", "Coach", "Businessperson", "Football Player", "Construction Worker",
    "Manager", "CEO", "Accountant", "Commander", "Firefighter", "Mover",
    "Software Developer", "Guard", "Baker", "Doctor", "Athlete", "Artist",
    "Dancer", "Mathematician", "Janitor", "Carpenter", "Mechanic", "Actor",
    "Handyman", "Musician", "Detective", "Politician", "Entrepreneur", "Model",
    "Opera Singer", "Lawyer", "Farmer", "Writer", "Librarian", "Soldier",
    "Real-Estate Developer", "Broker", "Scientist", "Butcher", "Electrician",
    "Prosecutor", "Banker", "Cook", "Hairdresser", "Prisoner", "Plumber",
    "Boxer", "Chess Player", "Priest", "Swimmer", "Tennis Player",
    "Supervisor", "Attendant", "Housekeeper", "Maid", "Producer", "Researcher",
    "Midwife", "Judge", "Bartender", "Economist",
    "Psychologist", "Theologian", "Salesperson", "Physician", "Sheriff",
    "Cashier", "Assistant", "Receptionist", "Editor", "Engineer", "Comedian",
    "Diplomat", "Guitarist", "Linguist", "Poet",
    "Laborer", "Teacher", "Delivery Driver", "Realtor", "Pilot", "Professor",
    "Historian", "Singer", "Secretary", "Auditor", "Counselor", 
    "Designer", "Journalist", "Dentist", "Analyst", "Nurse", "Tailor", "Waiter",
    "Architect", "Illustrator", "Clerk", "Police Officer", "Chef",
    "Photographer", "Cleaner", "Pharmacist", "Pianist", "Composer", "Handball Player",
    "Sociologist", "Audiologist", "Computer Programmer", "Dietitian", "DJ", "Driver",
    "Florist", "Graphic Designer", "Magician", "Makeup Artist", "Marine Biologist",
    "Nanny", "Optician", "Pastry Chef", "Sailor", "Social Worker",
    "Statistician", "Surgeon", "Technician", "Therapist", "Tour Guide", "Translator",
    "Vet", "Videographer", "Astronaut", "Biologist", "Civil Engineer",
    "Administrative Assistant", "Building Inspector", "Crane Operator", "Announcer",
    "Drafter", "Custodian", "Roofer", "PR Person", "Veterinarian", "Lab Tech",
    "Telemarketer", "Baseball Player", "Basketball Player", "Batter",
    "Goalie", "Pitcher", "Skater", "Skier",
    "Snowboarder", "Surfer", "Paramedic", "Mountain Rescue Worker",
    "Secret Service Agent", "Bodyguard", "Lifeguard", "SWAT Officer", "Security Officer",
    "Bounty Hunter", "Air Traffic Controller", "Flight Attendant", "Rocket Scientist",
    "Airplane Pilot", "Helicopter Pilot", "Train Conductor", "Taxi Driver", "Limousine Driver",
    "Race Car Driver", "Motorcycle Courier", "Bicycle Messenger", "Ship Captain",
    "Fisherman", "Animal Trainer", "Dog Walker", "Horse Trainer", "Zookeeper",
    "Park Ranger", "Wildlife Photographer", "Conservationist", "Ornithologist", 
    "Beekeeper", "Winemaker", "Brewer", "Sommelier", "Barista", "Chocolate Maker",
    "Candy Maker", "Cake Decorator", "Ice Cream Maker", "Cheese Maker", "Butler",
    "Valet", "Chauffeur", "Hotel Concierge", "Event Planner", "Wedding Planner",
    "Museum Curator", "Art Historian", "Gallery Owner", "Auctioneer",
    "Interior Designer", "Set Designer", "Costume Designer", "Fashion Designer",
    "Tattoo Artist", "Piercer", "Sculptor", "Glassblower",
    "Ceramic Artist", "Calligrapher", "Animator", "Video Editor", "Sound Engineer",
    "Lighting Technician", "Stage Manager", "Director", "Film Producer", "Critic",
    "News Anchor", "Sports Commentator", "Referee", "Martial Arts Instructor", "Personal Trainer", "Dance Instructor",
    "Ballet Teacher", "Clown Performer", "Acrobat", "Juggler",
    "Stunt Performer", "Fire Performer", "Musical Director", "Orchestra Conductor",
    "Rap Artist", "Pop Singer", "Rock Singer", "Jazz Musician", "Pianist", "Violinist",
    "Cellist", "Flutist", "Trumpet Player", "Saxophonist", "Drummer", "Guitarist",
    "Bass Guitarist", "Keyboardist", "Content Creator", "Photographer", "Drone Operator",
    "Documentary Filmmaker", "Author",
    "Editor", "Translator", "Interpreter", "Professor", "Teacher",
    "Librarian", "Archivist", "Historian", "Anthropologist", "Archaeologist",
    "Network Engineer", "Data Scientist", "Machine Learning Engineer",
    "AI Researcher", "Physicist", "Chemist", "Biologist", "Ecologist", "Geologist",
    "Oceanographer", "Meteorologist", "Pharmacist", "Nurse",
    "Dentist", "Optometrist", "Lawyer", "Judge", "Prosecutor",
    "Soldier", "Pilot", "Chef", "Florist", "Gardener", "Logger", "Miner",
    "Welder", "Blacksmith", "Bricklayer", "Plumber", "Mechanic", "Carpenter",
    "Tailor", "Seamstress", "Professional Gamer", "Chess Player",
    "Mountain Climber", "Diver", "Sailor", "Animal Shelter Worker", "Veterinarian"
]

def generate_custom_list(object_list, template="{object}"):
    """
    Generate a list of strings by applying a template to each profession.

    Args:
        object_list (list of str): List of profession names.
        template (str): Template string using "{object}" as a placeholder for the object.

    Returns:
        list of str: List with template applied to each profession.
    """
    result = []
    for obj in object_list:
        result.append(template.format(object=obj))
    return result

# Remove duplicates
all_prompts = list(sorted(set(profession_list)))
print(all_prompts)
print(f"Total Unique Prompts: {len(all_prompts)}")

# Process prompts in chunks with progress bar
for prompt_chunk in tqdm(list(chunk_list(all_prompts, chunk_size)), desc="Processing prompt chunks"):
    batch_prompts = []
    for template in prompt_templates:
        batch_prompts += generate_custom_list(prompt_chunk, template=template)
    print(batch_prompts)
    results = search_images(
        shard_index_paths=shard_indexes,
        shard_mapping_paths=shard_mappings,
        model_name="openai/clip-vit-large-patch14",
        prompts=batch_prompts,
        negative_prompts=["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
        top_k=1,#00_000, #6160,  #1540*4,
        similarity_threshold=0.15,
        copy_images=True,
        image_dir=r'F:\Thesis',
        output_dir=r'E:\Thesis\image_retrieval_images_2'
    )

['AI Researcher', 'Accountant', 'Acrobat', 'Actor', 'Administrative Assistant', 'Air Traffic Controller', 'Airplane Pilot', 'Analyst', 'Animal Shelter Worker', 'Animal Trainer', 'Animator', 'Announcer', 'Anthropologist', 'Archaeologist', 'Architect', 'Archivist', 'Art Historian', 'Artist', 'Assistant', 'Astronaut', 'Athlete', 'Attendant', 'Auctioneer', 'Audiologist', 'Auditor', 'Author', 'Baker', 'Ballet Teacher', 'Banker', 'Barber', 'Barista', 'Bartender', 'Baseball Player', 'Basketball Player', 'Bass Guitarist', 'Batter', 'Beekeeper', 'Bicycle Messenger', 'Biologist', 'Blacksmith', 'Bodyguard', 'Bounty Hunter', 'Boxer', 'Brewer', 'Bricklayer', 'Broker', 'Building Inspector', 'Businessperson', 'Butcher', 'Butler', 'CEO', 'Cake Decorator', 'Calligrapher', 'Candy Maker', 'Carpenter', 'Cashier', 'Cellist', 'Ceramic Artist', 'Chauffeur', 'Cheese Maker', 'Chef', 'Chemist', 'Chess Player', 'Chocolate Maker', 'Civil Engineer', 'Cleaner', 'Clerk', 'Clown Performer', 'Coach', 'Comedian', 'Comm

Processing prompt chunks:   0%|          | 0/55 [00:00<?, ?it/s]

['Male Person AI Researcher', 'Male Person Accountant', 'Male Person Acrobat', 'Male Person Actor', 'Male Person Administrative Assistant', 'Female Person AI Researcher', 'Female Person Accountant', 'Female Person Acrobat', 'Female Person Actor', 'Female Person Administrative Assistant']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Searching Group: 0000
Copying Images


Processing prompt chunks:   0%|          | 0/55 [00:20<?, ?it/s]


OSError: [WinError 1392] The file or directory is corrupted and unreadable: 'F:\\Thesis\\0000_images\\213913.jpg'

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel
import torch
import numpy as np
from itertools import combinations

# List of jobs derived from Papers & ChatGPT
clean_jobs = [
    "Barber", "Coach", "Businessperson", "Football Player", "Construction Worker",
    "Manager", "CEO", "Accountant", "Commander", "Firefighter", "Mover",
    "Software Developer", "Guard", "Baker", "Doctor", "Athlete", "Artist",
    "Dancer", "Mathematician", "Janitor", "Carpenter", "Mechanic", "Actor",
    "Handyman", "Musician", "Detective", "Politician", "Entrepreneur", "Model",
    "Opera Singer", "Lawyer", "Farmer", "Writer", "Librarian", "Soldier",
    "Real-Estate Developer", "Broker", "Scientist", "Butcher", "Electrician",
    "Prosecutor", "Banker", "Cook", "Hairdresser", "Prisoner", "Plumber",
    "Attorney", "Boxer", "Chess Player", "Priest", "Swimmer", "Tennis Player",
    "Supervisor", "Attendant", "Housekeeper", "Maid", "Producer", "Researcher",
    "Midwife", "Judge", "Umpire", "Bartender", "Economist", "Physicist",
    "Psychologist", "Theologian", "Salesperson", "Physician", "Sheriff",
    "Cashier", "Assistant", "Receptionist", "Editor", "Engineer", "Comedian",
    "Painter", "Diplomat", "Guitarist", "Linguist", "Poet",
    "Laborer", "Teacher", "Delivery Driver", "Realtor", "Pilot", "Professor",
    "Chemist", "Historian", "Singer", "Secretary", "Auditor", "Counselor", 
    "Designer", "Journalist", "Dentist", "Analyst", "Nurse", "Tailor", "Waiter",
    "Author", "Architect", "Illustrator", "Clerk", "Police Officer", "Chef",
    "Photographer", "Cleaner", "Pharmacist", "Pianist", "Composer", "Handball Player",
    "Sociologist", "Audiologist", "Computer Programmer", "Dietitian", "DJ", "Driver",
    "Florist", "Graphic Designer", "Magician", "Makeup Artist", "Marine Biologist",
    "Nanny", "Optician", "Pastry Chef", "Sailor", "Social Worker",
    "Statistician", "Surgeon", "Technician", "Therapist", "Tour Guide", "Translator",
    "Vet", "Videographer", "Zoologist", "Astronaut", "Biologist", "Civil Engineer",
    "Administrative Assistant", "Building Inspector", "Crane Operator", "Announcer",
    "Drafter", "Custodian", "Roofer", "PR Person", "Veterinarian", "Lab Tech",
    "Telemarketer", "Baseball Player", "Basketball Player", "Batter", "Catcher",
    "Goalie", "Pitcher", "Skateboarder", "Skater", "Skier", "Soccer Player",
    "Snowboarder", "Surfer", "Paramedic", "Mountain Rescue Worker", "Police Detective", 
    "Secret Service Agent", "Bodyguard", "Lifeguard", "SWAT Officer", "Security Officer",
    "Bounty Hunter", "Air Traffic Controller", "Flight Attendant", "Rocket Scientist",
    "Airplane Pilot", "Helicopter Pilot", "Train Conductor", "Taxi Driver", "Limousine Driver",
    "Race Car Driver", "Motorcycle Courier", "Bicycle Messenger", "Ship Captain",
    "Fisherman", "Animal Trainer", "Dog Walker", "Horse Trainer", "Zookeeper",
    "Park Ranger", "Wildlife Photographer", "Conservationist", "Ornithologist", 
    "Beekeeper", "Winemaker", "Brewer", "Sommelier", "Barista", "Chocolate Maker",
    "Candy Maker", "Cake Decorator", "Ice Cream Maker", "Cheese Maker", "Butler",
    "Valet", "Chauffeur", "Hotel Concierge", "Event Planner", "Wedding Planner",
    "Museum Curator", "Art Historian", "Gallery Owner", "Auctioneer",
    "Interior Designer", "Set Designer", "Costume Designer", "Fashion Designer",
    "Hairstylist", "Tattoo Artist", "Piercer", "Sculptor", "Glassblower",
    "Ceramic Artist", "Calligrapher", "Animator", "Video Editor", "Sound Engineer",
    "Lighting Technician", "Stage Manager", "Director", "Film Producer", "Film Critic",
    "Actor Extra", "Broadway Actor", "Choir Conductor", "DJ Performer", "News Anchor",
    "Sports Commentator", "Referee", "Soccer Coach", "Basketball Coach",
    "Tennis Coach", "Martial Arts Instructor", "Personal Trainer", "Dance Instructor",
    "Ballet Teacher", "Stage Actor", "Clown Performer", "Acrobat", "Juggler",
    "Stunt Performer", "Fire Performer", "Musical Director", "Orchestra Conductor",
    "Rap Artist", "Pop Singer", "Rock Singer", "Jazz Musician", "Pianist", "Violinist",
    "Cellist", "Flutist", "Trumpet Player", "Saxophonist", "Drummer", "Guitarist",
    "Bass Guitarist", "Keyboardist", "Content Creator", "Photographer", "Drone Operator",
    "Documentary Filmmaker", "Photojournalist", "Reporter", "Author", "Playwright",
    "Screenwriter", "Editor", "Translator", "Interpreter", "Professor", "Teacher",
    "Librarian", "Archivist", "Historian", "Anthropologist", "Archaeologist",
    "Museum Educator", "Mechanical Engineer", "Electrical Engineer", 
    "Software Engineer", "Network Engineer", "Data Scientist", "Machine Learning Engineer",
    "AI Researcher", "Physicist", "Chemist", "Biologist", "Ecologist", "Geologist",
    "Oceanographer", "Meteorologist", "Pharmacist", "Doctor", "Nurse",
    "Dentist", "Optometrist", "Lawyer", "Judge", "Prosecutor", "Fire Chief",
    "Soldier", "Pilot", "Chef", "Florist", "Gardener", "Logger", "Miner",
    "Welder", "Blacksmith", "Bricklayer", "Plumber", "Mechanic", "Carpenter",
    "Tailor", "Seamstress", "Professional Gamer", "Chess Player", "Horse Rider",
    "Mountain Climber", "Diver", "Sailor", "Animal Shelter Worker", "Veterinarian"
]

# Load CLIP tokenizer and text model
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")

# Correct way to get CLIP text embeddings
def get_text_embeddings(text_list):
    inputs = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        model_output = text_model(**inputs)
        # Pooling according to CLIP: mean of last_hidden_state weighted by attention_mask
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        masked_output = model_output.last_hidden_state * attention_mask
        embeddings = masked_output.sum(dim=1) / attention_mask.sum(dim=1)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)  # normalize
    return embeddings

embeddings = get_text_embeddings(clean_jobs)

# Compute pairwise cosine similarities efficiently
cos_sim_matrix = embeddings @ embeddings.T  # dot product of normalized embeddings

# Extract pairs
similarities = []
n = len(clean_jobs)
for i in range(n):
    for j in range(i+1, n):
        if clean_jobs[i] != clean_jobs[j]:
            similarities.append((clean_jobs[i], clean_jobs[j], cos_sim_matrix[i,j].item()))

# Sort by similarity descending
similarities_sorted = sorted(similarities, key=lambda x: x[2], reverse=True)

# Print top 20 most similar pairs
for pair in similarities_sorted[:30]:
    print(pair)

# Very similar jobs were removed by hand, and the new list used above when retrieivng images

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('Doctor', 'Physician', 0.8878103494644165)
('Dancer', 'Dance Instructor', 0.88323974609375)
('Writer', 'Author', 0.875404953956604)
('Pharmacist', 'Chemist', 0.8591217994689941)
('Chemist', 'Pharmacist', 0.8591217994689941)
('Musician', 'Guitarist', 0.8511295914649963)
('Musician', 'Guitarist', 0.8511295914649963)
('Designer', 'Fashion Designer', 0.8475048542022705)
('Umpire', 'Referee', 0.8434735536575317)
('Football Player', 'Baseball Player', 0.8413457274436951)
('Lawyer', 'Prosecutor', 0.839586615562439)
('Lawyer', 'Prosecutor', 0.839586615562439)
('Prosecutor', 'Lawyer', 0.839586615562439)
('Lawyer', 'Prosecutor', 0.839586615562439)
('Scientist', 'Chemist', 0.8377164602279663)
('Scientist', 'Biologist', 0.8374522924423218)
('Scientist', 'Biologist', 0.8374522924423218)
('Film Producer', 'Film Critic', 0.8373652696609497)
('Psychologist', 'Therapist', 0.8362483382225037)
('Optician', 'Optometrist', 0.8361414074897766)
('Barber', 'Hairdresser', 0.8345751762390137)
('Artist', 'Art H